# QuantJourney SDK - Economic Data & Macro Analysis

This notebook demonstrates macroeconomic data analysis:
- GDP and economic growth
- Inflation and CPI trends
- Unemployment rate
- Treasury yields and yield curve
- Economic dashboard

**API:** https://api.quantjourney.cloud

## Run Output

![03_economic_data_macro](../plots/03_economic_data_macro_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png" 

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# API Key authentication (recommended)
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. GDP Growth

In [ ]:
# Fetch GDP data
response = qj.fred.get_gdp()
gdp = response.get('value', response) if isinstance(response, dict) else response

if gdp:
    df_gdp = pd.DataFrame(gdp)
    df_gdp['date'] = pd.to_datetime(df_gdp['date'])
    df_gdp = df_gdp.sort_values('date')
    # FRED returns 'GDP' column, not 'value'
    df_gdp['qoq_growth'] = df_gdp['GDP'].pct_change() * 100
    df_gdp['yoy_growth'] = df_gdp['GDP'].pct_change(4) * 100
    
    print(f"GDP records: {len(df_gdp)}")
    print(f"Latest GDP: ${df_gdp['GDP'].iloc[-1]/1000:.2f} Trillion")
    df_gdp.tail()


In [ ]:
# Plot GDP and growth rates
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('US GDP (Trillions)', 'YoY Growth Rate'),
    vertical_spacing=0.1
)

fig.add_trace(
    go.Scatter(x=df_gdp['date'], y=df_gdp['GDP']/1000, name='GDP', fill='tozeroy'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=df_gdp['date'], y=df_gdp['yoy_growth'], name='YoY Growth'),
    row=2, col=1
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

fig.update_layout(
    title='US GDP Over Time',
    template='plotly_dark',
    height=600,
    showlegend=False
)
fig.show()


## 2. Inflation (CPI)

In [ ]:
# Fetch CPI data
response = qj.fred.get_cpi()
cpi = response.get('value', response) if isinstance(response, dict) else response

if cpi:
    df_cpi = pd.DataFrame(cpi)
    df_cpi['date'] = pd.to_datetime(df_cpi['date'])
    df_cpi = df_cpi.sort_values('date')
    # Find the CPI value column (CPIAUCSL or similar)
    value_col = [c for c in df_cpi.columns if c not in ['date', 'realtime_start', 'realtime_end']][0]
    df_cpi['yoy_inflation'] = df_cpi[value_col].pct_change(12) * 100
    df_cpi['mom_inflation'] = df_cpi[value_col].pct_change() * 100
    
    print(f"Value column: {value_col}")
    print(f"Latest CPI: {df_cpi[value_col].iloc[-1]:.1f}")
    print(f"YoY Inflation: {df_cpi['yoy_inflation'].iloc[-1]:.2f}%")
    print(f"Fed Target: 2.0%")


In [ ]:
# Plot inflation with Fed target
df_recent = df_cpi[df_cpi['date'] >= '2010-01-01']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_recent['date'],
    y=df_recent['yoy_inflation'],
    fill='tozeroy',
    name='Inflation Rate'
))

fig.add_hline(y=2, line_dash='dash', line_color='green', annotation_text='Fed Target 2%')
fig.add_hline(y=0, line_dash='solid', line_color='gray')

fig.update_layout(
    title='US Inflation Rate (CPI YoY Change)',
    yaxis_title='Inflation Rate (%)',
    xaxis_title='Date',
    template='plotly_dark',
    height=450
)
fig.show()


## 3. Unemployment Rate

In [ ]:
# Fetch unemployment data
response = qj.fred.get_unemployment_rate()
unemployment = response.get('value', response) if isinstance(response, dict) else response

if unemployment:
    df_unemp = pd.DataFrame(unemployment)
    df_unemp['date'] = pd.to_datetime(df_unemp['date'])
    df_unemp = df_unemp.sort_values('date')
    # Find the value column
    value_col = [c for c in df_unemp.columns if c not in ['date', 'realtime_start', 'realtime_end']][0]
    df_unemp['value'] = df_unemp[value_col]
    
    print(f"Current unemployment: {df_unemp['value'].iloc[-1]:.1f}%")
    print(f"Historical avg: {df_unemp['value'].mean():.1f}%")
    print(f"Historical min: {df_unemp['value'].min():.1f}%")
    print(f"Historical max: {df_unemp['value'].max():.1f}%")


In [ ]:
# Plot unemployment rate with recession highlighting
fig = px.area(
    df_unemp,
    x='date',
    y='value',
    title='US Unemployment Rate',
    template='plotly_dark'
)

fig.add_hline(y=df_unemp['value'].mean(), line_dash='dash', line_color='yellow',
              annotation_text=f"Historical Avg: {df_unemp['value'].mean():.1f}%")

fig.update_layout(
    yaxis_title='Unemployment Rate (%)',
    xaxis_title='Date',
    height=450
)
fig.show()


## 4. Treasury Yield Curve

In [ ]:
# Fetch treasury yields (only 2Y and 10Y available in FRED connector)
yields = {}

treasury_funcs = [
    ('2Y', 'get_treasury_2y'),
    ('10Y', 'get_treasury_10y'),
]

for tenor, method in treasury_funcs:
    try:
        response = getattr(qj.fred, method)()
        data = response.get('value', response) if isinstance(response, dict) else response
        if data:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            # Find the value column
            value_col = [c for c in df.columns if c not in ['date', 'realtime_start', 'realtime_end']][0]
            yields[tenor] = df.set_index('date')[value_col]
            print(f"✓ {tenor}: {df[value_col].iloc[-1]:.2f}%")
    except Exception as e:
        print(f"✗ {tenor}: {e}")

# Check for yield curve inversion
if '2Y' in yields and '10Y' in yields:
    spread = yields['10Y'].iloc[-1] - yields['2Y'].iloc[-1]
    status = "🔴 INVERTED" if spread < 0 else "🟢 Normal"
    print(f"\n2Y-10Y Spread: {spread:.2f}% ({status})")


In [ ]:
# Plot yield curves over time
df_yields = pd.DataFrame(yields)
df_yields.index = pd.to_datetime(df_yields.index)
df_yields = df_yields[df_yields.index >= '2020-01-01']

fig = px.line(
    df_yields,
    title='Treasury Yields (2020-Present)',
    template='plotly_dark'
)
fig.update_layout(
    yaxis_title='Yield (%)',
    xaxis_title='Date',
    legend_title='Tenor',
    height=500
)
fig.show()


In [ ]:
# Current yield curve shape
# Dropna to handle missing data
df_yields_clean = df_yields.dropna()
if len(df_yields_clean) > 0:
    current_yields = df_yields_clean.iloc[-1]
    tenor_order = [t for t in ['3M', '2Y', '5Y', '10Y', '30Y'] if t in current_yields.index]
    current_yields = current_yields[tenor_order]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=tenor_order,
        y=current_yields.values,
        mode='lines+markers',
        marker=dict(size=12),
        line=dict(width=3)
    ))

    fig.update_layout(
        title='Current Treasury Yield Curve',
        xaxis_title='Maturity',
        yaxis_title='Yield (%)',
        template='plotly_dark',
        height=400
    )
    fig.show()

    # Check for inversion
    if '2Y' in current_yields.index and '10Y' in current_yields.index:
        spread = current_yields['10Y'] - current_yields['2Y']
        status = "⚠️ INVERTED" if spread < 0 else "✓ Normal"
        print(f"\n2Y-10Y Spread: {spread*100:.0f}bp ({status})")
else:
    print("No data available for yield curve")


## 5. Fed Funds Rate

In [ ]:
# Fetch Fed Funds rate
response = qj.fred.get_effective_federal_funds_rate()
fed_funds = response.get('value', response) if isinstance(response, dict) else response

if fed_funds:
    df_ff = pd.DataFrame(fed_funds)
    df_ff['date'] = pd.to_datetime(df_ff['date'])
    df_ff = df_ff.sort_values('date')
    # Find value column
    value_col = [c for c in df_ff.columns if c not in ['date', 'realtime_start', 'realtime_end']][0]
    df_ff['value'] = df_ff[value_col]
    
    print(f"Current Fed Funds Rate: {df_ff['value'].iloc[-1]:.2f}%")


In [ ]:
# Plot Fed Funds rate history
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_ff['date'],
    y=df_ff['value'],
    fill='tozeroy',
    name='Fed Funds Rate'
))

fig.update_layout(
    title='Federal Funds Effective Rate',
    yaxis_title='Rate (%)',
    xaxis_title='Date',
    template='plotly_dark',
    height=450
)
fig.show()


## 6. Economic Dashboard

In [ ]:
# Create economic dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('GDP Growth (YoY)', 'Inflation (CPI YoY)', 
                    'Unemployment Rate', 'Fed Funds Rate'),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# GDP Growth
recent_gdp = df_gdp[df_gdp['date'] >= '2010-01-01']
fig.add_trace(
    go.Bar(x=recent_gdp['date'], y=recent_gdp['yoy_growth'], name='GDP Growth'),
    row=1, col=1
)

# Inflation
recent_cpi = df_cpi[df_cpi['date'] >= '2010-01-01']
fig.add_trace(
    go.Scatter(x=recent_cpi['date'], y=recent_cpi['yoy_inflation'], 
               fill='tozeroy', name='Inflation'),
    row=1, col=2
)

# Unemployment
recent_unemp = df_unemp[df_unemp['date'] >= '2010-01-01']
fig.add_trace(
    go.Scatter(x=recent_unemp['date'], y=recent_unemp['value'],
               fill='tozeroy', name='Unemployment'),
    row=2, col=1
)

# Fed Funds
recent_ff = df_ff[df_ff['date'] >= '2010-01-01']
fig.add_trace(
    go.Scatter(x=recent_ff['date'], y=recent_ff['value'],
               fill='tozeroy', name='Fed Funds'),
    row=2, col=2
)

fig.update_layout(
    title='US Economic Dashboard (2010-Present)',
    template='plotly_dark',
    height=700,
    showlegend=False
)
fig.show()


In [ ]:
# Summary table
print("\n" + "="*50)
print("ECONOMIC INDICATORS SUMMARY")
print("="*50)
print(f"\nGDP (Latest):        ${df_gdp['GDP'].iloc[-1]/1000:.2f} Trillion")
print(f"GDP Growth (YoY):    {df_gdp['yoy_growth'].iloc[-1]:.1f}%")
print(f"Inflation (YoY):     {df_cpi['yoy_inflation'].iloc[-1]:.1f}%")
print(f"Unemployment:        {df_unemp['value'].iloc[-1]:.1f}%")
print(f"Fed Funds Rate:      {df_ff['value'].iloc[-1]:.2f}%")
if '10Y' in yields and len(yields['10Y']) > 0:
    print(f"10Y Treasury:        {yields['10Y'].iloc[-1]:.2f}%")
else:
    print(f"10Y Treasury:        N/A")


## Summary

FRED connector methods used:
- `qj.fred.get_gdp()` - GDP data
- `qj.fred.get_cpi()` - Inflation (CPI)
- `qj.fred.get_unemployment_rate()` - Unemployment
- `qj.fred.get_effective_federal_funds_rate()` - Fed Funds
- `qj.fred.get_Xy_treasury_rate()` - Treasury yields